### Preprocessing METABRIC dataset to get P gene x tumor binary matrix, sutype labels, and molecular interactions network 

In [1]:
#load data Charlotte (last edited - 11/25)
import data_preprocessing as dp

mutation_file = "data_mutations.txt"
cna_file = "data_cna.txt"
clinical_sample_file = "data_clinical_sample.txt"
clinical_patient_file = "data_clinical_patient.txt"

#get raw P0
mutation_matrix = dp.load_mutation_data(mutation_file)
cna_matrix = dp.load_cna_data(cna_file)
#combined P0
P0 = dp.integrate_mutation_cna(mutation_matrix, cna_matrix)

#align and P0 subtype labels
sample_to_patient = dp.load_sample_to_patient_map(clinical_sample_file)
patient_subtypes = dp.load_patient_subtypes(clinical_patient_file)
sample_subtypes = dp.get_sample_subtypes(sample_to_patient, patient_subtypes)
aligned_P0, aligned_labels = dp.align_labels_with_matrix(sample_subtypes, P0)



In [2]:
# Check if it looks right - dont need to run 
print("Alteration matrix (P) shape:", aligned_P0.shape)
print("Subtype label vector length:", len(aligned_labels))
print("Sample IDs in matrix head:", aligned_P0.index[:5].tolist())
print("Gene features in matrix head:", aligned_P0.columns[:5].tolist())
print("Subtype labels head:", aligned_labels.head())


Alteration matrix (P) shape: (524, 96)
Subtype label vector length: 524
Sample IDs in matrix head: ['MB-0010', 'MB-0039', 'MB-0056', 'MB-0062', 'MB-0093']
Gene features in matrix head: ['AFF2', 'AGMO', 'AHNAK2', 'AKAP9', 'ALK']
Subtype labels head: MB-0010     LumB
MB-0039     LumB
MB-0056     LumB
MB-0062    Basal
MB-0093     LumB
Name: PATIENT_ID, dtype: object


In [3]:

#ask if saveing results to CSV and uploading to github would help for rest of implementation 
aligned_P0.to_csv("aligned_matrix_P.csv")
aligned_labels.to_csv("aligned_subtype_labels.csv")

In [4]:
#parse and initalize molecular network 
molIN, nodes, df_filteredEdges = dp.setup_molecular_interaction_network("Regulon BRCA - Breast Invasive Carcinoma.cx", aligned_P0)

#getting the edge feature matrix representation 
w_init_sd = 0.01 # from Zhange et al github 


Shared genes: 94
Genes in your panel but missing in network: {'KMT2C', 'MLLT4'}
Genes in network but not in your panel: ['SNORD33', 'MT3', 'EIF3J', 'ARMCX5', 'RPS15AP10', 'PSMD8', 'UAP1L1', 'PTPRO', 'CUL4B', 'TTTY16']
  SourceGene TargetGene
0      NR2E3      HSFX2
1      NR2E3       ESR1
2      NR2E3       AZU1
3      NR2E3       MST1
4      NR2E3       COA7


In [5]:
#filenames for the cancer related pathway gene sets 
pathway_features= {
    'HALLMARK_APOPTOSIS': 'pathway_apoptosis',
    'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION': 'pathway_emt',
    'HALLMARK_GLYCOLYSIS': 'pathway_glycolysis',
    'HALLMARK_IL6_JAK_STAT3_SIGNALING': 'pathway_jak_stat',
    'HALLMARK_INFLAMMATORY_RESPONSE': 'pathway_inflammatory',
    'HALLMARK_P53_PATHWAY': 'pathway_p53',
    'HALLMARK_TGF_BETA_SIGNALING': 'pathway_tgfb',
    'HALLMARK_WNT_BETA_CATENIN_SIGNALING': 'pathway_wnt',
    'KEGG_ADHERENS_JUNCTION': 'pathway_adherens_junction',
    'KEGG_BASE_EXCISION_REPAIR': 'pathway_ber',
    'KEGG_B_CELL_RECEPTOR_SIGNALING_PATHWAY': 'pathway_bcr',
    'KEGG_CELL_CYCLE': 'pathway_cell_cycle',
    'KEGG_CYTOKINE_CYTOKINE_RECEPTOR_INTERACTION': 'pathway_cytokine',
    'KEGG_ECM_RECEPTOR_INTERACTION': 'pathway_ecm',
    'KEGG_ERBB_SIGNALING_PATHWAY': 'pathway_erbb',
    'KEGG_FOCAL_ADHESION': 'pathway_focal_adhesion',
    'KEGG_MAPK_SIGNALING_PATHWAY': 'pathway_mapk',
    'KEGG_MISMATCH_REPAIR': 'pathway_mmr',
    'KEGG_MTOR_SIGNALING_PATHWAY': 'pathway_mtor',
    'KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY': 'pathway_nk_cytotoxicity',
    'KEGG_NUCLEOTIDE_EXCISION_REPAIR': 'pathway_ner',
    'KEGG_PATHWAYS_IN_CANCER': 'pathway_cancer',
    'KEGG_PPAR_SIGNALING_PATHWAY': 'pathway_ppar',
    'KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY': 'pathway_tcr',
    'KEGG_VEGF_SIGNALING_PATHWAY': 'pathway_vegf',
    'REACTOME_TELOMERE_MAINTENANCE': 'pathway_telomere',
}
dir = "/Users/emtgk/python/03712/FinalProject/CancerRelatedPathways"

geneSets, pathwayFeatInfo = dp.load_cancer_related_gene_sets(pathway_features, dir)

Found 26 TSV files
26 pathways loaded successfully


In [7]:
# Phase 2: Extract edge attributes
df_edges = dp.extract_edge_attributes(molIN, df_filteredEdges)

# Phase 3: Compute gene-level features
mutationRate = dp.compute_mutation_rates(aligned_P0)
subtype_pval = dp.compute_subtype_association(aligned_P0, aligned_labels)

# Phase 5: Build complete edge feature matrix
edge_feature_matrix = dp.edge_feature_matrix(
    df_edges, 
    mutationRate, 
    subtype_pval,
    geneSets, 
    pathwayFeatInfo,  
    aligned_P0
)



Extracted Mechanism of Action and Likelihood for 33 edges
Mechanism of Action: 33 values present
Likelihood Score: 33 values present
Computing subtype association for 96 genes across 7 subtypes
  Edges: 33
  Features: 45


In [8]:
edge_feature_matrix.to_csv("Molecular Interaction Network.csv")